# Install requirements and imports

In [28]:
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 7.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [33]:
with open("The-Universe-in-Your-Hand.txt", encoding="utf-8") as f:
    text = f.read()

print("length of book is: ", len(text))
print ("==============================")
print(text[:100])

chars = sorted(list(set(text)))
vocab_size = len(chars)
print ("==============================")
print(''.join(chars))
print("vocab size is: ", vocab_size)

length of book is:  619804
Foreword





Before we start, there are two things I would like to share with you.

The first is a 

 !&()*,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz|£©­°Öçíîö–—‘’“”•
vocab size is:  94


# Choosing encoding/decoding

In [ ]:
# stoi = { ch:i for i,ch in enumerate(chars) }
# itos = { i:ch for i,ch in enumerate(chars) }
# encode = lambda s: [stoi[c] for c in s] #string to integers
# decode = lambda l: ''.join([itos[i] for i in l]) #integers to string

# # could also use tiktoken (what chatgpt uses)

In [ ]:
import tiktoken

# Choose an encoding (e.g., "gpt2" or "cl100k_base" for GPT-3.5/4)
encoding = tiktoken.get_encoding("gpt2")

# Encode: string to list of integers (tokens)
encoded = encoding.encode(text)

vocab_size = encoding.n_vocab  # Use tiktoken's vocab size

# Decode: list of integers (tokens) back to string
decoded = encoding.decode(encoded)

print(encoded)
print(decoded)

# makes an instance of the bigramlanguagemodel class with the vocab size of the book
# intitializes the internal layers (embedding table, etc.) with the given vocab_size
m = BigramLanguageModel(vocab_size)

[16351, 4775, 628, 628, 198, 198, 8421, 356, 923, 11, 612, 389, 734, 1243, 314, 561, 588, 284, 2648, 351, 345, 13, 198, 198, 464, 717, 318, 257, 6991, 11, 262, 1218, 318, 281, 20505, 13, 198, 198, 464, 6991, 318, 326, 262, 1492, 4909, 691, 530, 16022, 13, 198, 198, 4342, 340, 318, 25, 628, 198, 198, 36, 28, 23209, 17, 628, 628, 198, 198, 464, 20505, 11, 616, 20505, 11, 318, 326, 287, 428, 1492, 314, 481, 407, 2666, 597, 7183, 2157, 13, 198, 198, 1639, 389, 546, 284, 923, 257, 7002, 832, 262, 6881, 355, 340, 318, 7247, 416, 3783, 1909, 13, 632, 318, 616, 25420, 4901, 326, 356, 460, 477, 1833, 428, 3404, 13, 198, 198, 1870, 326, 7002, 6140, 257, 845, 890, 835, 422, 1363, 11, 319, 262, 584, 1735, 286, 262, 995, 13, 628, 628, 198, 198, 7841, 1881, 628, 198, 464, 39972, 628, 628, 198, 198, 16, 930, 317, 25083, 31824, 628, 628, 198, 198, 28070, 3511, 319, 257, 1290, 8272, 31513, 7022, 319, 257, 5814, 11, 6279, 1203, 3931, 1755, 13, 383, 7346, 9151, 318, 355, 991, 355, 257, 13546, 13, 5514, 2

In [48]:
print(m)

BigramLanguageModel(
  (token_embedding_table): Embedding(50257, 50257)
)


In [46]:
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encoded, dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([146567]) torch.int64
tensor([16351,  4775,   628,   628,   198,   198,  8421,   356,   923,    11,
          612,   389,   734,  1243,   314,   561,   588,   284,  2648,   351,
          345,    13,   198,   198,   464,   717,   318,   257,  6991,    11,
          262,  1218,   318,   281, 20505,    13,   198,   198,   464,  6991,
          318,   326,   262,  1492,  4909,   691,   530, 16022,    13,   198,
          198,  4342,   340,   318,    25,   628,   198,   198,    36,    28,
        23209,    17,   628,   628,   198,   198,   464, 20505,    11,   616,
        20505,    11,   318,   326,   287,   428,  1492,   314,   481,   407,
         2666,   597,  7183,  2157,    13,   198,   198,  1639,   389,   546,
          284,   923,   257,  7002,   832,   262,  6881,   355,   340,   318,
         7247,   416,  3783,  1909,    13,   632,   318,   616, 25420,  4901,
          326,   356,   460,   477,  1833,   428,  3404,    13,   198,   198,
         1870,   326,  7002,  6

In [38]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [39]:
block_size = 8
train_data[:block_size+1]

tensor([16351,  4775,   628,   628,   198,   198,  8421,   356,   923])

In [40]:
block_size = 8
train_data[:block_size+1]
# will make a prediction at each of these positions for the following character

tensor([16351,  4775,   628,   628,   198,   198,  8421,   356,   923])

In [41]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([16351]) the target: 4775
when input is tensor([16351,  4775]) the target: 628
when input is tensor([16351,  4775,   628]) the target: 628
when input is tensor([16351,  4775,   628,   628]) the target: 198
when input is tensor([16351,  4775,   628,   628,   198]) the target: 198
when input is tensor([16351,  4775,   628,   628,   198,   198]) the target: 8421
when input is tensor([16351,  4775,   628,   628,   198,   198,  8421]) the target: 356
when input is tensor([16351,  4775,   628,   628,   198,   198,  8421,   356]) the target: 923


In [42]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1492,   588,   383, 11950,   287,  3406,  7157,    11],
        [  262,  2042,  7604,   345,   389,  4964,   318, 27382],
        [   13,   921,   766,   340,  5836,   826,   287,  2166],
        [ 3025,  5485,   345,   389,  4379,    11,   428,  9664]])
targets:
torch.Size([4, 8])
tensor([[  588,   383, 11950,   287,  3406,  7157,    11,   340],
        [ 2042,  7604,   345,   389,  4964,   318, 27382,    11],
        [  921,   766,   340,  5836,   826,   287,  2166,   286],
        [ 5485,   345,   389,  4379,    11,   428,  9664,   543]])
----
when input is [1492] the target: 588
when input is [1492, 588] the target: 383
when input is [1492, 588, 383] the target: 11950
when input is [1492, 588, 383, 11950] the target: 287
when input is [1492, 588, 383, 11950, 287] the target: 3406
when input is [1492, 588, 383, 11950, 287, 3406] the target: 7157
when input is [1492, 588, 383, 11950, 287, 3406, 7157] the target: 11
when input is [1492, 588, 383, 1

In [43]:
print(xb) # our input to the transformer

tensor([[ 1492,   588,   383, 11950,   287,  3406,  7157,    11],
        [  262,  2042,  7604,   345,   389,  4964,   318, 27382],
        [   13,   921,   766,   340,  5836,   826,   287,  2166],
        [ 3025,  5485,   345,   389,  4379,    11,   428,  9664]])


In [44]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# 0 is the element for a new line character, so we will start with that 
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

IndexError: index out of range in self

loss was about log(-1/95) which is expected based on number of characters. it's not very good yet!

notice that the generated text is absolutely random :)

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(100000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

5.06643533706665


In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


cfJF3c*LA’°;VQ7uZ–­MI
EyW©&uQ9”O; Z5n
­YwjHN
X;Q3d(0’em•eYK9deP‘4/°Ö£)ILpsy:UHD1Ö9&°h2Kno6°3hl?dCÖçÖçzS.9GEmK;2K”­mW(12Dgqj“uíçz AIO:=WVM?Rt-xpA:suko—N*/u5u‘•wlp,2Q&wR£nLZg7“D:d1Jcaöx0­yVx(’3£JcöA=tqEd8”8of4|I|ne6.‘­yuö4I­Ub’2H-I2‘I6*&QI|DXT3V-î6ygnnDy—nnCI|ob)/RX°’4—ayZFK9C6v.NMNz3fjx’çq!W;!Z•*SSB;Fd,V
dbCÖPP’UqPSH(U’4I5|lDjxg”OQ­jMoi?JKD1Ug.1”y2î|p–k—J/a!yVC,,Nj;1g(nfqAQ£yEHKmiM““S—NW©|kM–ZîChkCUFaZÖC‘­gc
­HK“‘MIyPVKWL“­y:“Q7
g6R(p7!v0“pv—Rk:J1oC/7FQ5©M3mK)S*£hK4l•A’hta”cSN”2ÖXsN/!H/h-­*xu)wu.


the pieces are not talking to each other yet!

wei = wei.masked_fill(trill ==0, float('-inf))
this is the decoder block so you have the triangular structure

you could also delete it to have the encoder block

both are allowed, attention doesn't care; supports 'arbitrary connection between nodes'

what is the diff between attention and cross-attention?
if keys queries and the values come from source x, then this would be self-attending nodes. 

in cross-attention: you can still have queries from x, but:
keys and values could come from an external source (e.g. some of the encoder blocks that include context that we'd like to condition on)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

with open("The-Universe-in-Your-Hand.txt", encoding="utf-8") as f:
    text = f.read()
    
# here are all the unique characters that occur in this text
# chars = sorted(list(set(text)))
# vocab_size = len(chars)

# create a mapping from characters to integers
# stoi = { ch:i for i,ch in enumerate(chars) }
# itos = { i:ch for i,ch in enumerate(chars) }
# encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
# decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


# alternate embedding
encoding = tiktoken.get_encoding("gpt2")
# Encode your book text
encoded = encoding.encode(text)
# Decode back to string
decoded = encoding.decode(encoded)
vocab_size = encoding.n_vocab

# Train and test splits
# data = torch.tensor(encode(text), dtype=torch.long)
data = torch.tensor(encoded, dtype=torch.long)

n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(encoding.decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


6.684497 M parameters
step 0: train loss 11.0039, val loss 10.9342
step 100: train loss 6.3343, val loss 8.2781
step 200: train loss 5.9813, val loss 8.4289
step 300: train loss 5.7069, val loss 8.5945
step 400: train loss 5.5089, val loss 8.5931
step 500: train loss 5.3033, val loss 8.6524
step 600: train loss 5.1529, val loss 8.6805
step 700: train loss 5.0264, val loss 8.6978
step 800: train loss 4.9078, val loss 8.5234
step 900: train loss 4.8183, val loss 8.9327
step 1000: train loss 4.6969, val loss 8.7959
step 1100: train loss 4.6377, val loss 8.8093
step 1200: train loss 4.5785, val loss 8.9342
step 1300: train loss 4.4900, val loss 8.8150
step 1400: train loss 4.4413, val loss 8.8330
step 1500: train loss 4.3605, val loss 8.7476
step 1600: train loss 4.2770, val loss 8.7642
step 1700: train loss 4.2681, val loss 8.8577
step 1800: train loss 4.1979, val loss 9.0274
step 1900: train loss 4.1623, val loss 9.1687
step 2000: train loss 4.1095, val loss 9.1129
step 2100: train loss 